# Polysemantic tokens: one transformer token per hidden unit, then the 4 × 128 MLP head

Same log-sum-exp pooling as the 4 × 128 run (29 layers → 15 windows of 3 layers, stride 2), giving a 15 × 3,584 matrix. Each of the 3,584 hidden units becomes one **polysemantic token**: its 15-value trajectory across the pooled layers, standardized with source-train statistics, embedded by a shared Linear(15 → 128) plus a learned per-neuron positional embedding. A [CLS] token and the 3,584 neuron tokens pass through 2 pre-LN transformer blocks (d 128, 4 heads, FFN 512, GELU); [CLS] is projected to a constant 512-d embedding, which goes into the same residual Mish 4 × 128 MLP head.

Same six-source split, BCE, batch 128, AdamW (lr 1e-3, weight decay 1e-4), seed 42, best-source-validation-AUROC checkpoint, threshold calibration and held-out TriviaQA evaluation as the 4 × 128 run; 10 epochs.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout 8c03a8e
!pip install -q -r requirements-kaggle.txt

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
OUTPUT_DIR = Path('/kaggle/working/sala_g14_6src_polysemantic_4x128')
command = [sys.executable, '/kaggle/working/SALA/sala_polysemantic.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(OUTPUT_DIR), '--bleurt-threshold', '0.5', '--epochs-erm', '10', '--seed', '42', '--train-device', 'cuda:0', '--model-dim', '128', '--heads', '4', '--blocks', '2', '--ffn-dim', '512', '--embedding-dim', '512', '--hidden-dim', '128', '--hidden-layers', '4']
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
results = pd.read_csv(OUTPUT_DIR / 'results.csv')
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
print('Polysemantic tokens:', summary['polysemantic_tokens'], 'of length', summary['token_length'], '| parameters:', summary['parameters'], '| best epoch:', summary['best_epoch'])
print('Source val AUROC:', summary['best_source_validation_auroc_during_training'])
print('Source test:', json.dumps(summary['source_test_metrics'], indent=2))
print('Target (TriviaQA):', json.dumps(summary['target_metrics'], indent=2))
display(results.head())